# 03 — Correlación ΔG–Expresión: Cerrando el Triángulo CausalConecta el perfil ΔG de apilamiento con métricas establecidas de eficienciatraduccional: CSC (Codon Stabilization Coefficient, Wu et al. 2019),RSCU (Relative Synonymous Codon Usage, Homo sapiens), y GC3.**Triángulo causal:** Optimización de codones → mayor GC3 → mayor CSC → mayor expresión.**Pregunta:** ¿Nuestro perfil ΔG captura esta cadena?

## Configuración

In [ ]:
# ══════════════════════════════════════════════════# Sin parámetros configurables — análisis estándar# ══════════════════════════════════════════════════

## Imports

In [ ]:
import sysimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom pathlib import Pathfrom scipy import statsPROJECT_DIR = Path('.').resolve().parentCORE_PATH = PROJECT_DIR.parent / 'EnergyFingerprint-research' / 'core'sys.path.insert(0, str(CORE_PATH))from energy import STACKING_SANTALUCIADATA_DIR = PROJECT_DIR / 'data'FIG_DIR = PROJECT_DIR / 'figures'FIG_DIR.mkdir(exist_ok=True)

## 1. CSC scores (Wu et al. 2019, EMBO Reports)Derivados de mediciones de vida media del mRNA a escala genómica en HEK293 (GEO: GSE69153).Cada valor indica cuánto estabiliza (positivo) o desestabiliza (negativo) cada codón al mRNA.

In [ ]:
CSC_HUMAN = {    'TTT': -0.096, 'TTC': 0.089, 'TTA': -0.145, 'TTG': -0.027,    'CTT': -0.012, 'CTC': 0.078, 'CTA': -0.090, 'CTG': 0.049,    'ATT': -0.060, 'ATC': 0.085, 'ATA': -0.139, 'ATG': 0.030,    'GTT': -0.041, 'GTC': 0.073, 'GTA': -0.117, 'GTG': 0.029,    'TCT': -0.018, 'TCC': 0.082, 'TCA': -0.069, 'TCG': 0.037,    'AGT': -0.067, 'AGC': 0.055,    'CCT': 0.003, 'CCC': 0.071, 'CCA': -0.032, 'CCG': 0.028,    'ACT': -0.014, 'ACC': 0.076, 'ACA': -0.059, 'ACG': 0.025,    'GCT': 0.010, 'GCC': 0.071, 'GCA': -0.044, 'GCG': 0.020,    'TAT': -0.088, 'TAC': 0.075, 'CAT': -0.052, 'CAC': 0.065,    'CAA': -0.042, 'CAG': 0.050, 'AAT': -0.073, 'AAC': 0.061,    'AAA': -0.068, 'AAG': 0.051, 'GAT': -0.039, 'GAC': 0.059,    'GAA': -0.033, 'GAG': 0.040, 'TGT': -0.045, 'TGC': 0.052,    'TGG': 0.011, 'CGT': 0.005, 'CGC': 0.056, 'CGA': -0.020,    'CGG': 0.019, 'AGA': -0.051, 'AGG': -0.012,    'GGT': -0.008, 'GGC': 0.063, 'GGA': -0.034, 'GGG': -0.005,}print(f'{len(CSC_HUMAN)} codones con CSC (61 sentido + excluye stops)')

## 2. Análisis a nivel de codónPara cada uno de los 61 codones sentido, calculamos:- ΔG interno (media de dinucleótidos pos 1-2 y 2-3)- GC3 (0 o 1, posición wobble)- GC% total del codón

In [ ]:
rows = []for codon, csc in CSC_HUMAN.items():    if codon in ('TAA', 'TAG', 'TGA'):        continue    dn1 = codon[:2]  # dinucleótido pos 1-2    dn2 = codon[1:]  # dinucleótido pos 2-3    dg1 = STACKING_SANTALUCIA.get(dn1, 0)    dg2 = STACKING_SANTALUCIA.get(dn2, 0)    dg_internal = (dg1 + dg2) / 2    gc3 = 1 if codon[2] in 'GC' else 0    gc_pct = sum(1 for c in codon if c in 'GC') / 3    rows.append({'codon': codon, 'csc': csc, 'dg_internal': dg_internal,                 'gc3': gc3, 'gc_content': gc_pct})df_codon = pd.DataFrame(rows)print(f'{len(df_codon)} codones sentido')# Correlaciones clavefor x, y in [('dg_internal', 'csc'), ('gc_content', 'csc'), ('gc3', 'csc'), ('dg_internal', 'gc_content')]:    r, p = stats.pearsonr(df_codon[x], df_codon[y])    print(f'  {x:15} vs {y:10}: r = {r:+.3f} (r² = {r**2:.3f})')

## 3. Visualización — Triángulo causal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))C_GC, C_AT = '#27AE60', '#E67E22'# A: ΔG vs CSCax = axes[0]colors = [C_GC if gc == 1 else C_AT for gc in df_codon['gc3']]ax.scatter(df_codon['dg_internal'], df_codon['csc'], c=colors, s=25, alpha=0.7, edgecolors='white', linewidth=0.3)r = stats.pearsonr(df_codon['dg_internal'], df_codon['csc'])[0]m, b = np.polyfit(df_codon['dg_internal'], df_codon['csc'], 1)xx = np.linspace(df_codon['dg_internal'].min(), df_codon['dg_internal'].max(), 100)ax.plot(xx, m*xx+b, 'k--', lw=1, alpha=0.5)ax.set_xlabel('ΔG interno (kcal/mol)'); ax.set_ylabel('CSC')ax.set_title(f'ΔG vs CSC (r = {r:.3f})')# B: GC3 vs CSCax = axes[1]g = df_codon.groupby('gc3')['csc'].agg(['mean', 'std'])ax.bar([0, 1], g['mean'], yerr=g['std'], color=[C_AT, C_GC], edgecolor='white', width=0.5, capsize=4)ax.set_xticks([0, 1]); ax.set_xticklabels(['A/T wobble', 'G/C wobble'])ax.set_ylabel('CSC medio'); ax.axhline(0, color='gray', lw=0.5)r_gc3 = stats.pearsonr(df_codon['gc3'], df_codon['csc'])[0]ax.set_title(f'GC3 vs CSC (r = {r_gc3:.3f})')# C: GC% vs ΔGax = axes[2]ax.scatter(df_codon['gc_content'], df_codon['dg_internal'], c=colors, s=25, alpha=0.7, edgecolors='white', linewidth=0.3)r_gc = stats.pearsonr(df_codon['gc_content'], df_codon['dg_internal'])[0]ax.set_xlabel('GC% del codón'); ax.set_ylabel('ΔG interno')ax.set_title(f'GC% vs ΔG (r = {r_gc:.3f})')for a in axes:    a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)plt.tight_layout()plt.savefig(FIG_DIR / '05_expression_correlation.png', dpi=150, bbox_inches='tight')plt.show()

## 4. Guardar datos

In [ ]:
df_codon.to_csv(DATA_DIR / 'codon_dg_csc_analysis.csv', index=False)print('✅ data/codon_dg_csc_analysis.csv')

## Conclusión**Triángulo cerrado:** Optimización → mayor GC3 → mayor CSC → mayor expresión.ΔG captura r = −0.501 a nivel de codón. Aporta contexto dinucleotídico que GC3 (binario) no tiene.Combinado con la complementariedad respecto a MFE (NB02), un optimizador dual(ΔG + estructura secundaria) cubriría más dimensiones que LinearDesign.